In [3]:
import sys, os, glob, json, re, collections
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pretty_midi

UTENTE = 'martinopaoloni-ux'
BRANCH = 'main'            # il branch dove si trova src/midi_convert.py

!rm -rf /content/MozartAI
!git clone -b {BRANCH} https://github.com/{UTENTE}/MozartAI.git /content/MozartAI
percorsi = glob.glob('/content/MozartAI/**/midi_convert.py', recursive=True)
assert percorsi, ("midi_convert.py non trovato. File .py presenti nel repo: "
                  + str(glob.glob('/content/MozartAI/**/*.py', recursive=True)))
sys.path.insert(0, os.path.dirname(percorsi[0]))
from midi_convert import tokens_to_midi, tokens_to_notes, analizza_token
print("Import ok, file usato:", percorsi[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/MozartAI'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 111 (delta 46), reused 9 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 7.66 MiB | 9.60 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Import ok, file usato: /content/MozartAI/src/midi_convert.py


In [4]:
VOCAB = '/content/drive/MyDrive/spartiti/m1/vocab.json'
vocab = list(json.load(open(VOCAB, encoding='utf-8')))
print(len(vocab), "simboli")

famiglie = collections.defaultdict(list)
for t in vocab:
    famiglie[t.split('-')[0]].append(t)
for nome, lista in sorted(famiglie.items(), key=lambda x: -len(x[1])):
    print(f"{nome:15s} {len(lista):5d}   es.: {lista[:3]}")

1781 simboli
note             1384   es.: ['note-A#2_eighth', 'note-A#2_half', 'note-A#2_quarter']
gracenote         210   es.: ['gracenote-A#3_eighth', 'gracenote-A#3_sixteenth', 'gracenote-A#4_eighth']
multirest          98   es.: ['multirest-1', 'multirest-10', 'multirest-100']
timeSignature      39   es.: ['timeSignature-1/2', 'timeSignature-1/4', 'timeSignature-11/4']
rest               24   es.: ['rest-eighth', 'rest-eighth.', 'rest-eighth..']
keySignature       14   es.: ['keySignature-AM', 'keySignature-AbM', 'keySignature-BM']
clef               10   es.: ['clef-C1', 'clef-C2', 'clef-C3']
barline             1   es.: ['barline']
tie                 1   es.: ['tie']


In [5]:
non_ok = [t for t in vocab if analizza_token(t)['tipo'] in ('sconosciuto', 'malformato')]
print(len(non_ok), "simboli non riconosciuti dal convertitore")
print(non_ok[:60])

alterazioni = collections.Counter()
for t in vocab:
    m = re.match(r'^(?:note|gracenote)-[A-G]([^0-9-]*)-?\d', t)
    if m:
        alterazioni[m.group(1) or '(nessuna)'] += 1
print("Alterazioni nelle note:", alterazioni)

suffissi = collections.Counter(t.split('_', 1)[1] for t in vocab if t.startswith('note-') and '_' in t)
print("Durate e suffissi:", suffissi.most_common(40))

0 simboli non riconosciuti dal convertitore
[]
Alterazioni nelle note: Counter({'(nessuna)': 781, '#': 424, 'b': 389})
Durate e suffissi: [('sixteenth', 92), ('quarter', 90), ('eighth', 88), ('half', 85), ('thirty_second', 81), ('quarter.', 80), ('eighth.', 80), ('whole', 74), ('half.', 71), ('sixteenth.', 66), ('half_fermata', 52), ('sixty_fourth', 45), ('whole.', 45), ('double_whole', 43), ('quarter_fermata', 41), ('quarter..', 40), ('whole_fermata', 39), ('eighth..', 39), ('quarter._fermata', 28), ('half._fermata', 27), ('eighth_fermata', 26), ('double_whole.', 25), ('double_whole_fermata', 22), ('quadruple_whole', 20), ('thirty_second.', 20), ('quadruple_whole_fermata', 11), ('eighth._fermata', 10), ('whole._fermata', 10), ('quadruple_whole.', 8), ('half..', 7), ('sixteenth_fermata', 6), ('hundred_twenty_eighth', 5), ('sixteenth._fermata', 3), ('double_whole._fermata', 2), ('sixty_fourth.', 1), ('quarter.._fermata', 1), ('sixteenth..', 1)]


In [6]:
# 1) Il tempo cambia i secondi, non i battiti
for tempo in (60, 120):
    pm = tokens_to_midi("note-C4_quarter".split(), tempo=tempo)
    print(f"tempo {tempo}: la nota dura {pm.instruments[0].notes[0].end:.2f} secondi")

# 2) L'armatura (parametro applica_armatura)
seq = "keySignature-FM note-B4_quarter".split()
print("senza armatura:", tokens_to_notes(seq)[0][0], "| con armatura:", tokens_to_notes(seq, applica_armatura=True)[0][0])

# 3) La legatura somma due note uguali, ma ignora due note diverse
print(tokens_to_notes("note-A4_half tie note-A4_quarter".split()))
avvisi = []
print(tokens_to_notes("note-D4_quarter tie note-B4_quarter".split(), avvisi=avvisi), avvisi)

tempo 60: la nota dura 1.00 secondi
tempo 120: la nota dura 0.50 secondi
senza armatura: 71 | con armatura: 70
[[69, 0.0, 3.0]]
[[62, 0.0, 1.0], [71, 1.0, 1.0]] ['Legatura tra note diverse ignorata (prima di note-B4_quarter)']


In [7]:
SEMITONI = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}
ALTERAZIONI = {'': 0, '#': 1, 'b': -1, '##': 2, 'bb': -2}

def nome_a_midi(nome):
    """'D5' -> 74,  'Bb3' -> 58,  'F#5' -> 78"""
    lettera = nome[0]               # la prima lettera: la nota
    ottava = int(nome[-1])          # l'ultimo carattere: il numero dell'ottava
    alterazione = nome[1:-1]        # quello che sta in mezzo: '', '#', 'b'...
    return 12 * (ottava + 1) + SEMITONI[lettera] + ALTERAZIONI[alterazione]

# Controlli: se non stampa errori e arriva a "ok", la funzione è giusta
assert nome_a_midi('C4') == 60
assert nome_a_midi('A4') == 69
assert nome_a_midi('D5') == 74
assert nome_a_midi('Bb3') == 58
assert nome_a_midi('F#5') == 78
print("ok")

ok


In [8]:
DURATE = {'whole': 4, 'half': 2, 'quarter': 1, 'eighth': 0.5, 'sixteenth': 0.25}

def durata(nome, punti=0):
    """('quarter', 1) -> 1.5   (ogni punto aggiunge metà del valore precedente)"""
    return DURATE[nome] * (2 - 0.5 ** punti)

assert durata('quarter') == 1
assert durata('quarter', 1) == 1.5
assert durata('half', 2) == 3.5
assert durata('eighth', 1) == 0.75
print("ok")

ok


In [9]:
melodia = "clef-G2 keySignature-FM timeSignature-4/4 note-F4_quarter note-G4_quarter note-A4_half rest-quarter note-F4_quarter. note-G4_eighth barline".split()
for altezza, inizio, d in tokens_to_notes(melodia):
    print(f"altezza MIDI {altezza}  inizia al battito {inizio}  dura {d} battiti")

altezza MIDI 65  inizia al battito 0.0  dura 1.0 battiti
altezza MIDI 67  inizia al battito 1.0  dura 1.0 battiti
altezza MIDI 69  inizia al battito 2.0  dura 2.0 battiti
altezza MIDI 65  inizia al battito 5.0  dura 1.5 battiti
altezza MIDI 67  inizia al battito 6.5  dura 0.5 battiti


In [10]:
import tarfile, io
import pretty_midi
from IPython.display import Audio

ARCHIVE = '/content/drive/MyDrive/MozartAI/data/primus.tgz'
esempi, parziali = [], {}
with tarfile.open(ARCHIVE, 'r:gz') as tar:
    for m in tar:
        nome = os.path.basename(m.name)
        if not m.isfile() or nome.startswith('._'):
            continue
        cartella = os.path.dirname(m.name)
        if nome.endswith('.semantic'):
            parziali.setdefault(cartella, {})['sem'] = tar.extractfile(m).read().decode('utf-8').split()
        elif nome.endswith('.mid'):
            parziali.setdefault(cartella, {})['mid'] = tar.extractfile(m).read()
        d = parziali.get(cartella, {})
        if 'sem' in d and 'mid' in d:
            esempi.append(parziali.pop(cartella))
            if len(esempi) == 5:
                break
print(len(esempi), "esempi con .semantic e .mid")

def altezze(pm):
    return [n.pitch for n in sorted(pm.instruments[0].notes, key=lambda n: (n.start, n.pitch))]

for e in esempi:
    avvisi = []
    mio = tokens_to_midi(e['sem'], tempo=100, avvisi=avvisi)
    originale = pretty_midi.PrettyMIDI(io.BytesIO(e['mid']))
    print("MIO      :", altezze(mio))
    print("ORIGINALE:", altezze(originale))
    print("uguali:", altezze(mio) == altezze(originale), "| avvisi:", avvisi, "\n")

mio.write('/content/prova.mid')
Audio(mio.synthesize(fs=22050), rate=22050)

5 esempi con .semantic e .mid
MIO      : [72, 70, 69, 67, 69, 70, 69, 67, 65, 69, 70, 72, 70]
ORIGINALE: [72, 71, 69, 67, 69, 71, 69, 67, 65, 69, 71, 72, 71]
uguali: False | avvisi: ['2 appoggiature (gracenote) ignorate'] 

MIO      : [67, 72, 76, 77, 76, 74, 72, 71, 72, 72, 72, 74, 76, 77, 76, 74, 72, 74, 76, 74]
ORIGINALE: [67, 72, 76, 77, 76, 74, 72, 71, 72, 72, 72, 74, 76, 77, 76, 74, 72, 74, 76, 74]
uguali: True | avvisi: [] 

MIO      : [71, 71, 71, 69, 69, 69, 74, 72, 71, 72, 74, 71, 72, 69]
ORIGINALE: [71, 71, 71, 69, 69, 69, 74, 72, 71, 72, 74, 71, 72, 69]
uguali: True | avvisi: [] 

MIO      : [70, 75, 75, 70, 70, 67, 67, 72, 72, 77, 77, 72, 72, 74, 75, 75, 74, 70, 70]
ORIGINALE: [71, 76, 76, 71, 71, 67, 67, 72, 72, 77, 77, 72, 72, 74, 76, 76, 74, 71, 71]
uguali: False | avvisi: [] 

MIO      : [67, 69, 70, 72, 74, 75, 78, 79, 81, 82, 78, 79, 74, 77]
ORIGINALE: [67, 69, 71, 72, 74, 76, 78, 79, 81, 83, 78, 79, 74, 77]
uguali: False | avvisi: [] 



In [11]:
ducks = "clef-G2 keySignature-DM timeSignature-C note-B4_quarter gracenote-D4_eighth note-D4_quarter tie note-B4_quarter barline note-B4_quarter note-A4_quarter note-A4_quarter rest-quarter barline".split()
prima = "clef-F4 keySignature-GM timeSignature-2/4 gracenote-A4_sixteenth timeSignature-4/4 barline gracenote-G4_sixteenth rest-quarter note-D#4_eighth. barline timeSignature-2/2 note-G4_eighth.. note-G4_eighth.. note-E4_eighth barline barline tie timeSignature-3/2 tie multirest-33".split()

for nome, seq in [('Five little ducks', ducks), ('Prima parte', prima), ('vuota', []), ('spazzatura', ['x', None, 42, 'note-C99_quarter'])]:
    avvisi = []
    pm = tokens_to_midi(seq, avvisi=avvisi)
    print(f"{nome:18s} note: {len(pm.instruments[0].notes)}  avvisi: {avvisi}")

Five little ducks  note: 6  avvisi: ['Legatura tra note diverse ignorata (prima di note-B4_quarter)', '1 appoggiature (gracenote) ignorate']
Prima parte        note: 4  avvisi: ['2 appoggiature (gracenote) ignorate']
vuota              note: 0  avvisi: []
spazzatura         note: 0  avvisi: ['Simbolo sconosciuto saltato: x', 'Simbolo sconosciuto saltato: None', 'Simbolo sconosciuto saltato: 42', "Nota fuori dall'intervallo MIDI saltata: note-C99_quarter"]
